## Import de Bibliotecas

In [5]:
import os
import random
import time
import numpy as np

# Função de caraga das instâncias

### Transforma os arquivos Knapsack em dicionários, com: 
- capacidade ('c') 
- peso dos objetos ('w') 
- lucro de cada objeto ('p') 
- seleção otimizada de objetos ('s')

In [18]:
def ler_instancia_dicionario(nome: str, pasta: str = f'utils/dados_mochila') -> dict:
    def ler_vetor(sufixo):
        caminho = os.path.join(pasta, nome, f"{nome}_{sufixo}.txt")
        if not os.path.exists(caminho):
            caminho = os.path.join(pasta, f'{nome}_{sufixo}.txt')
        with open(caminho, 'r') as f:
            return np.array([float(x) for x in f.read().strip().split()])

    capacidade = ler_vetor('c')[0]
    pesos = ler_vetor('w')
    lucros = ler_vetor('p')

    otimo_vetor = ler_vetor('s').astype(int)
    lucro_otimo = float(np.sum(lucros * otimo_vetor))

    return {
        'nome': nome,
        'capacidade': capacidade,
        'pesos': pesos,
        'lucros': lucros,
        'n_itens': len(pesos),
        'lucro_otimo': lucro_otimo
    }

# Criação de Indivíduo e População

In [ ]:
def criar_individuo(n_itens: int, rng: np.random.Generator) -> np.ndarray:
    return rng.integers(0, 2, size=n_itens)

def criar_populacao(tam_pop: int, n_itens: int, rng: np.random.Generator) -> np.ndarray:
    return rng.integers(0, 2, size=(tam_pop, n_itens))

# Fitness

- **ind * pesos** multiplica posição por posição, então só os itens com 1 contam, e np.sum soma o resultado.
- **peso_total** e **lucro_total** somam o peso e o lucro dos itens que estão na mochila.
- **e_viavel** diz se o peso cabe na capacidade, e serve só para conferir a solução.
- **fitness** é o lucro menos (excesso de peso × maior razão lucro/peso), então estourar a capacidade nunca compensa.
- Exemplo: peso 183, capacidade 165, lucro 233 dão fitness 233 − 18 × 4 = 161.
- Fitness negativo é normal em indivíduos muito inviáveis, e funciona com torneio, mas não com roleta.

    Estrategia: PENALIZACAO.
    fitness = lucro - penalidade * excesso_de_peso

    A penalidade usa a maior razao lucro/peso da instancia, o que garante que
    estourar a capacidade nunca compensa. Se preferir, troque por reparo ou
    fitness zero para inviaveis e compare nos experimentos.

In [ ]:
def peso_total(ind, inst):
    return float(np.sum(ind * inst['pesos']))

def lucro_total(ind, inst):
    return float(np.sum(ind * inst['lucros']))

def e_viavel(ind, inst):
    return peso_total(ind, inst) <= inst['capacidade']

def fitness(ind, inst):
    lucro = lucro_total(ind, inst)
    excesso = max(0, peso_total(ind, inst) - inst['capacidade'])
    penalidade = np.max(inst['lucros']/inst['pesos'])
    return lucro - excesso * penalidade

# Operadores

In [ ]:
def selecao_torneio(pop, fits, k, rng):
    indices = rng.integers(0, len(pop), size=k)
    melhor_indice = indices[np.argmax(fits[indices])]
    return pop[melhor_indice].copy()

def cruzamento(pai1, pai2, taxa_cruzamento, rng):
    numero = rng.random()
    if numero >= taxa_cruzamento:
        return pai1.copy(), pai2.copy()
    ponto = rng.integers(1, len(pai1))
    filho1 = np.concatenate((pai1[:ponto], pai2[ponto:]))
    filho2 = np.concatenate((pai2[:ponto], pai1[ponto:]))
    return filho1, filho2

def mutacao(ind, taxa_mutacao, rng):
    filho = ind.copy()
    mascara = rng.random(len(filho)) < taxa_mutacao
    filho[mascara] = 1 - filho[mascara]
    return filho
        

In [19]:
if __name__ == '__main__':
    inst = [ler_instancia_dicionario(f"p0{i}") for i in range(1, 6)]
    for instance in inst:
        print(f'Instância: {instance["nome"]}\n'
              f' Capacidade: {instance["capacidade"]}\n'
              f' Pesos: {instance["pesos"]}\n'
              f' Lucros: {instance["lucros"]}\n'
              f' Número de Itens: {instance["n_itens"]}\n'
              f' Lucro Ótimo: {instance["lucro_otimo"]}')

Instância: p01
 Capacidade: 165.0
 Pesos: [23. 31. 29. 44. 53. 38. 63. 85. 89. 82.]
 Lucros: [92. 57. 49. 68. 60. 43. 67. 84. 87. 72.]
 Número de Itens: 10
 Lucro Ótimo: 309.0
Instância: p02
 Capacidade: 26.0
 Pesos: [12.  7. 11.  8.  9.]
 Lucros: [24. 13. 23. 15. 16.]
 Número de Itens: 5
 Lucro Ótimo: 51.0
Instância: p03
 Capacidade: 190.0
 Pesos: [56. 59. 80. 64. 75. 17.]
 Lucros: [50. 50. 64. 46. 50.  5.]
 Número de Itens: 6
 Lucro Ótimo: 150.0
Instância: p04
 Capacidade: 50.0
 Pesos: [31. 10. 20. 19.  4.  3.  6.]
 Lucros: [70. 20. 39. 37.  7.  5. 10.]
 Número de Itens: 7
 Lucro Ótimo: 107.0
Instância: p05
 Capacidade: 104.0
 Pesos: [25. 35. 45.  5. 25.  3.  2.  2.]
 Lucros: [350. 400. 450.  20.  70.   8.   5.   5.]
 Número de Itens: 8
 Lucro Ótimo: 900.0
